# Revision R2 — Reviewer #4 experiments

Three additions, all resume-safe and all writing ONLY to `results/revision_R2/`
on Drive. The submitted `results/v2/` and first-round `results/revision_R1/`
are never modified.

| Section | Reviewer point | What it does |
|---------|----------------|--------------|
| **A** | **R4-P2** | 30-seed stability on the single pair jackhhao→advbench (BERT, ModernBERT, MiniLM+LR) |
| **B** | **R4-P3** | DAN 60-char length confound: 8 detectors + a length-only baseline, capped vs uncapped |
| **C** | **R4-P1** | Inference-memory profiler for the 8 configs (isolated subprocess, batch=1) |

**Compute note.** The heavy cost is the two fine-tuned transformers. Section A
is `2 encoders × 30` short fine-tunes on ~930 rows (cheap). Section B adds
`2 encoders × 7 conditions × 10 seeds` **capped** fine-tunes — the bulk, on the
order of tens of GPU-hours on a T4/L4. Everything else (TF-IDF, embedding+head,
length-only, memory profiler) is minutes.

**Resume.** Every cell is resume-safe. If a session disconnects, just re-run the
cell: completed `(Method, Dataset, Seed, HyperParams)` rows (and completed
`Method`s in the profiler) are skipped. Heavy sections can each run in their own
Colab session.

## 0. Colab bootstrap

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Location of this repository on your Google Drive; edit if you cloned it elsewhere.
CANDIDATES = [
    '/content/drive/MyDrive/llm-jailbreak-cross-dataset-benchmark',
]
DRIVE_PROJECT = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DRIVE_PROJECT, f'Project not found in {CANDIDATES}'
LOCAL_PROJECT = '/content/llm-jailbreak-cross-dataset-benchmark'
DRIVE_RESULTS_R2 = f'{DRIVE_PROJECT}/results/revision_R2'
print('Drive project :', DRIVE_PROJECT)

In [ ]:
import shutil
# Fresh local copy of the code each session (fast local disk).
if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)
shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT)
os.chdir(LOCAL_PROJECT)
os.makedirs(DRIVE_RESULTS_R2, exist_ok=True)
print('Working dir:', os.getcwd())
print('Results ->', DRIVE_RESULTS_R2)

In [ ]:
# ModernBERT needs transformers>=4.48; DeBERTa/tokenizers need sentencepiece.
!pip install -q -U "transformers>=4.48" sentencepiece
!pip install -q torch sentence-transformers scikit-learn scipy pandas numpy xgboost psutil

In [ ]:
import torch, transformers
print('transformers:', transformers.__version__)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 0b. Provenance + frozen-split persistence

Pin package versions, torch/CUDA, and the HuggingFace model revision hashes to
`results/revision_R2/provenance.txt` (so the reused R1 `orig` rows are provably
comparable to the new capped rows), and define `sync_splits_to_drive()` — which
copies any newly-built frozen manifests from the fast local disk back to Drive,
so a resumed session reuses the SAME frozen splits instead of reconstructing
them.

In [ ]:
import os, shutil, sys
import importlib.metadata as _im
import torch


def sync_splits_to_drive(local=LOCAL_PROJECT, drive=DRIVE_PROJECT, verbose=True):
    """Copy newly-built frozen manifests local->Drive (build_manifests is
    deterministic, so this is a provenance/safety guarantee). Existing Drive
    files with the same size are left untouched; only missing/changed files are
    copied. Cheap validation: Drive must end with >= as many manifests as local."""
    src_root = os.path.join(local, 'splits')
    dst_root = os.path.join(drive, 'splits')
    copied = 0
    for dirpath, _d, files in os.walk(src_root):
        rel = os.path.relpath(dirpath, src_root)
        dst_dir = os.path.join(dst_root, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for fn in files:
            s = os.path.join(dirpath, fn); d = os.path.join(dst_dir, fn)
            if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
                continue
            shutil.copy2(s, d); copied += 1
    _cnt = lambda root: sum(1 for dp, _x, fs in os.walk(root)
                            for f in fs if f == 'manifest.json')
    n_local, n_drive = _cnt(src_root), _cnt(dst_root)
    if verbose:
        print(f'[sync] copied {copied} split file(s) local->Drive; '
              f'manifests local={n_local} drive={n_drive}')
    assert n_drive >= n_local, 'Drive is missing manifests after sync!'


def write_provenance(path=None):
    """APPEND a timestamped provenance section (one per session) so multi-session
    runs are all logged instead of the file being overwritten each time. An HF
    SHA lookup failure prints a LOUD warning rather than silently continuing."""
    import datetime
    path = path or os.path.join(DRIVE_RESULTS_R2, 'provenance.txt')
    os.makedirs(os.path.dirname(path), exist_ok=True)
    session = 1
    if os.path.isfile(path):
        with open(path) as f:
            session += sum(1 for ln in f if ln.startswith('# === session '))
    ts = datetime.datetime.now().isoformat(timespec='seconds')
    lines = [f'# === session {session} @ {ts} ===',
             '# Revision R2 provenance', f'python: {sys.version.split()[0]}',
             f'torch: {torch.__version__}  cuda: {torch.version.cuda}  '
             f'cuda_available: {torch.cuda.is_available()}']
    if torch.cuda.is_available():
        lines.append(f'gpu: {torch.cuda.get_device_name(0)}')
    for pkg in ['torch', 'transformers', 'sentence-transformers', 'scikit-learn',
                'scipy', 'pandas', 'numpy', 'xgboost', 'tokenizers',
                'huggingface-hub', 'psutil']:
        try:
            lines.append(f'{pkg}: {_im.version(pkg)}')
        except Exception:
            lines.append(f'{pkg}: (not installed)')
    lines.append('# HF model revisions (resolved commit sha on main)')
    models = ['bert-base-uncased', 'answerdotai/ModernBERT-base',
              'sentence-transformers/all-MiniLM-L6-v2',
              'Snowflake/snowflake-arctic-embed-l']
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        for m in models:
            try:
                lines.append(f'{m}: {api.model_info(m).sha}')
            except Exception as e:
                lines.append(f'{m}: (lookup FAILED: {e})')
                print(f'WARNING: HF sha lookup FAILED for {m}: {e} '
                      f'(model revision NOT pinned in provenance.txt)')
    except Exception as e:
        lines.append(f'(huggingface_hub unavailable: {e})')
        print(f'WARNING: huggingface_hub unavailable, NO model SHAs pinned: {e}')
    with open(path, 'a') as f:
        f.write('\n'.join(lines) + '\n\n')
    print('\n'.join(lines)); print(f'-> {path} (appended as session {session})')


write_provenance()

## 1. Build the extended-seed manifests

Section B (length confound) uses the 10 seeds `SEEDS_EXTENDED` on DAN
in-distribution and the six DAN-facing cross pairs. Seeds `42/123/456` already
ship in `splits/`; we build the seven extended seeds here (deterministic,
idempotent — same content R1 built). Section A's 30-seed runner builds its own
single pair for the twenty extra seeds automatically, so nothing extra is needed
for it here.

In [ ]:
NEW_SEEDS = '789 1011 1213 1415 1617 1819 2021'
!python experiments/build_manifests.py --seeds {NEW_SEEDS}

In [ ]:
sync_splits_to_drive()  # persist the extended-seed manifests to Drive

## A. R4-P2 — 30-seed stability (jackhhao → advbench)

Builds the pair's manifests for the twenty new seeds (existing ten untouched),
then runs BERT, ModernBERT and MiniLM+LR for all 30 seeds. Per-seed F1 rows go
to `stability_30seed.csv`; aggregation (mean / CI / permutation test) is done in
analysis, not here. Resume-safe — safe to split across sessions.

In [ ]:
# Persist the 20 NEW 30-seed pair manifests to Drive BEFORE the long
# run consumes the session, so a mid-run disconnect cannot lose the
# frozen splits. --build_only builds (skip-if-present) then exits.
!python experiments/revision_R2/run_30seed.py --build_only \
  --methods BERT ModernBERT MiniLM+LR
sync_splits_to_drive()  # persist the new pair manifests to Drive

In [ ]:
!python experiments/revision_R2/run_30seed.py \
  --result_file {DRIVE_RESULTS_R2}/stability_30seed.csv \
  --methods BERT ModernBERT MiniLM+LR

In [ ]:
sync_splits_to_drive()  # persist the 20 new 30-seed pair manifests to Drive

## B. R4-P3 — DAN 60-char length confound

Truncation is applied at load time (after manifest membership + leakage
removal). Only DAN is truncated, both classes, to the first 60 code points. The
`60`-vs-`orig` condition is stored in `HyperParams` and echoed in `Notes`; the
`Dataset` tag stays plain.

The uncapped (`orig`) numbers for the **8 detectors** are byte-identical to
results already in `results/revision_R1`, so they are **imported** (step B.0)
rather than recomputed. Only the capped detector rows and the LengthOnly
baseline (which is not in R1, so it needs both `orig` and `60`) are new compute
here.

The collision audit runs with the first capped cell and writes
`length_confound_collisions.csv` (prefix duplicates on the actual asymmetrically
capped inputs: within-train, within-val, train-vs-test, val-vs-test).

### B.0 Import the uncapped (`orig`) detector rows from R1

Imports the 8 detectors' uncapped rows (10 seeds x 7 conditions) verbatim from
`results/revision_R1`, tagged `max_chars="orig"`, into `length_confound.csv`.
Fails loudly if any expected source row is missing or duplicated, so the
orig-vs-60 table can never be silently incomplete.

In [ ]:
!python experiments/revision_R2/import_orig_rows.py \
  --result_file {DRIVE_RESULTS_R2}/length_confound.csv

### B.1 Capped detectors (TF-IDF, embeddings) — 60 only

The `orig` rows for these six detectors were imported in B.0 from
`results/revision_R1`; here we compute ONLY the 60-char-capped rows. The
collision audit runs once here.

In [ ]:
!python experiments/revision_R2/run_length_confound.py \
  --result_file {DRIVE_RESULTS_R2}/length_confound.csv \
  --collision_file {DRIVE_RESULTS_R2}/length_confound_collisions.csv \
  --methods TF-IDF+LR TF-IDF+SVM MiniLM+LR MiniLM+XGBoost Arctic+RF BERT-frozen+LR \
  --max_chars 60

### B.1b LengthOnly baseline — `orig` AND `60` (new compute)

The single-feature (character-count) LengthOnly baseline is not in R1, so it
genuinely needs both length settings computed here.

In [ ]:
!python experiments/revision_R2/run_length_confound.py \
  --result_file {DRIVE_RESULTS_R2}/length_confound.csv \
  --collision_file {DRIVE_RESULTS_R2}/length_confound_collisions.csv \
  --methods LengthOnly+LR --max_chars orig 60 --skip_audit

### B.2 BERT — capped only (own session OK)
The uncapped BERT rows for these conditions were imported in B.0 from
`results/revision_R1`; here we add only the capped rows.

In [ ]:
!python experiments/revision_R2/run_length_confound.py \
  --result_file {DRIVE_RESULTS_R2}/length_confound.csv \
  --collision_file {DRIVE_RESULTS_R2}/length_confound_collisions.csv \
  --methods BERT --max_chars 60 --skip_audit

### B.3 ModernBERT — capped only (own session OK)

In [ ]:
!python experiments/revision_R2/run_length_confound.py \
  --result_file {DRIVE_RESULTS_R2}/length_confound.csv \
  --collision_file {DRIVE_RESULTS_R2}/length_confound_collisions.csv \
  --methods ModernBERT --max_chars 60 --skip_audit

## C. R4-P1 — inference memory profiler

Profiles the full deployment artifact for each of the 8 configs at batch=1, each
in an isolated subprocess (peaks can't bleed), with a warm-up forward. Reports
host RSS peak and, on CUDA, `max_memory_allocated` / `max_memory_reserved`, plus
total params and on-disk checkpoint size. Seconds per config; resume-safe by
`Method`.

In [ ]:
!python experiments/revision_R2/mem_profile.py \
  --result_file {DRIVE_RESULTS_R2}/memory_profile.csv \
  --seqlen 512 --batch 1

## D. Sanity check the written results

In [ ]:
import pandas as pd, glob, os
for f in sorted(glob.glob(f'{DRIVE_RESULTS_R2}/*.csv')):
    d = pd.read_csv(f)
    name = os.path.basename(f)
    if 'Method' in d.columns:
        print(name, '| rows:', len(d), '| methods:', sorted(d['Method'].unique()))
    else:
        print(name, '| rows:', len(d), '| cols:', list(d.columns))

## E. Coverage assertion (gate before using numbers)

Fails LOUDLY (non-zero exit) if any of the three result CSVs is incomplete or has an OOM/failed row. Run this last; a red cell here means the paper data is NOT ready.

In [ ]:
!python experiments/revision_R2/check_coverage.py --results_dir {DRIVE_RESULTS_R2}